# CogAttention — Rule Shift

**Track:** Attention — Attention Shifting
**Benchmark:** CogAttention v1.0
**Task:** shifting

---

## Methodology

Tests attention shifting through rule-switch classification — the model classifies words by one rule, then must switch to a new rule mid-task. Measures perseveration errors and attentional residue. Based on WCST (Monsell, 2003) and Task Interference (EMNLP 2024).

### Cognitive Science Grounding

- **Wisconsin Card Sorting Test / Task Switching** (Monsell, 2003): classify items by one rule, then the rule changes mid-task
- 60-70% of post-switch errors are perseveration (old rule) or attentional residue (pre-switch context bleeding in), not random hallucination
- Causal self-attention mechanically anchors to earlier context, capping flexibility

### Difficulty Scaling

Easy: 4 post-switch items, 1 rule change | Medium: 6 items | Hard: 10 items | Expert: 15 items | Frontier: 20 items, triple rule change

### Scoring

One assertion per classified item — checks correct rule application. Post-switch errors are classified as perseveration, residue, or random.

All instances are procedurally generated from a seed with programmatic ground truth.
No static datasets. 7 layers of contamination resistance including canary strings,
zero lexical overlap (Semantic NIAH), and seed-based regeneration.

---

`<!-- COGATTENTION-BENCH-CANARY-C9FC65F1ADF9 -->`


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2: Imports + Inline Helpers
# CogAttention — Attention Shifting
# ══════════════════════════════════════════════════════════════════════

import kaggle_benchmarks as kbench

import json
import re

def extract_answer_block(response):
    for pat in [r"ANSWER:\s*(.*)", r"Answer:\s*(.*)", r"answer:\s*(.*)"]:
        match = re.search(pat, response, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return response.strip()

def extract_numbered_answers(response):
    answer_block = extract_answer_block(response)
    results = {}
    matches = re.findall(
        r"(\d+)\s*[.):\-]\s*(.+?)(?=\n\d+\s*[.):\-]|\Z)",
        answer_block, re.DOTALL,
    )
    for num, val in matches:
        results[num] = val.strip().rstrip(".")
    return results

def extract_list_items(response):
    answer_block = extract_answer_block(response)
    bullets = re.findall(r"[-\u2022]\s*(.+?)(?:\n|$)", answer_block)
    if bullets:
        return [b.strip().rstrip(".") for b in bullets]
    numeric_items = re.findall(
        r'[\$]?\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s*(?:\xb0[CF]|mg/L|%|\$))?',
        answer_block,
    )
    if numeric_items and len(numeric_items) >= 2:
        return [x.strip() for x in numeric_items]
    if "," in answer_block:
        items = [x.strip().rstrip(".") for x in answer_block.split(",")]
        return [x for x in items if x]
    lines = [l.strip().rstrip(".") for l in answer_block.split("\n") if l.strip()]
    return lines if lines else ([answer_block] if answer_block else [])

def extract_person_item_pairs(response):
    answer_block = extract_answer_block(response)
    results = {}
    for pat in [
        r"[-\u2022]?\s*(\w+)\s*:\s*(.+?)(?:\n|$)",
        r"[-\u2022]?\s*(\w+)\s+holds?\s+(?:a\s+)?(.+?)(?:\n|$)",
    ]:
        matches = re.findall(pat, answer_block, re.IGNORECASE)
        if matches:
            for name, item in matches:
                results[name.strip()] = item.strip().rstrip(".")
            break
    return results

def fuzzy_value_match(predicted, gold):
    pred_clean = re.sub(r"\s+", " ", predicted.strip().lower())
    gold_clean = re.sub(r"\s+", " ", gold.strip().lower())
    if pred_clean == gold_clean:
        return True
    if gold_clean in pred_clean:
        return True
    try:
        pred_num = float(re.sub(r"[,$%\xb0]", "", predicted))
        gold_num = float(re.sub(r"[,$%\xb0]", "", gold))
        return pred_num == gold_num
    except (ValueError, TypeError):
        pass
    return False

def _escape_for_regex(s):
    return re.escape(s).replace(r"\ ", r"\s+")


def run_assertions_shifting(response, gold, kbench):
    for idx_str, gold_val in gold["answers"].items():
        pattern = rf"(?i){re.escape(idx_str)}\s*[.):\-]\s*.*{_escape_for_regex(gold_val)}"
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"Item {idx_str} should be classified as '{gold_val}'"
        )


print("CogAttention helpers loaded")
print(f"Task types: ['shifting']")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3: Task Definitions + Embedded Dataset
# ══════════════════════════════════════════════════════════════════════


@kbench.task(name="cogattention_shifting")
def cogattention_shifting(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention shifting task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_shifting(response, gold, kbench)


# ── Embedded dataset ──────────────────────────────────────────────────
DATASET = json.loads(r'''
[
 {
  "task_id": "shifting_easy_000",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. candle → ?\n2. mirror → ?\n3. shovel → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n4. penguin → ?\n5. eagle → ?\n6. wrench → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"object\", \"2\": \"object\", \"3\": \"object\", \"4\": \"late\", \"5\": \"early\", \"6\": \"late\"}}"
 },
 {
  "task_id": "shifting_easy_001",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. mango → ?\n2. banana → ?\n3. basket → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by CATEGORY: animal, food, or object\n\n4. turnip → ?\n5. tiger → ?\n6. orange → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"early\", \"2\": \"early\", \"3\": \"early\", \"4\": \"food\", \"5\": \"animal\", \"6\": \"food\"}}"
 },
 {
  "task_id": "shifting_easy_002",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. penguin → ?\n2. giraffe → ?\n3. shovel → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by CATEGORY: animal, food, or object\n\n4. wrench → ?\n5. pepper → ?\n6. candle → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"late\", \"2\": \"early\", \"3\": \"late\", \"4\": \"object\", \"5\": \"food\", \"6\": \"object\"}}"
 },
 {
  "task_id": "shifting_easy_003",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. needle → ?\n2. shovel → ?\n3. walnut → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n4. hammer → ?\n5. orange → ?\n6. mirror → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"object\", \"2\": \"object\", \"3\": \"food\", \"4\": \"early\", \"5\": \"late\", \"6\": \"early\"}}"
 },
 {
  "task_id": "shifting_easy_004",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. needle → ?\n2. tiger → ?\n3. falcon → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n4. mirror → ?\n5. orange → ?\n6. giraffe → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"object\", \"2\": \"animal\", \"3\": \"animal\", \"4\": \"early\", \"5\": \"late\", \"6\": \"early\"}}"
 },
 {
  "task_id": "shifting_easy_005",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. giraffe → ?\n2. candle → ?\n3. shovel → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n4. hammer → ?\n5. wrench → ?\n6. banana → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"animal\", \"2\": \"object\", \"3\": \"object\", \"4\": \"early\", \"5\": \"late\", \"6\": \"early\"}}"
 },
 {
  "task_id": "shifting_easy_006",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. parrot → ?\n2. wrench → ?\n3. cherry → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n4. needle → ?\n5. turnip → ?\n6. penguin → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"animal\", \"2\": \"object\", \"3\": \"food\", \"4\": \"late\", \"5\": \"late\", \"6\": \"late\"}}"
 },
 {
  "task_id": "shifting_easy_007",
  "task_type": "shifting",
  "difficulty": "Easy",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. hammer → ?\n2. eagle → ?\n3. mirror → ?\n\n--- RULE CHANGE ---\nNEW RULE: Classify each word by CATEGORY: animal, food, or object\n\n4. shovel → ?\n5. falcon → ?\n6. walnut → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"early\", \"2\": \"early\", \"3\": \"early\", \"4\": \"object\", \"5\": \"animal\", \"6\": \"food\"}}"
 },
 {
  "task_id": "shifting_medium_008",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. cherry → ?\n2. needle → ?\n3. falcon → ?\n4. eagle → ?\n\nRule change: Classify each word by CATEGORY: animal, food, or object\n\n5. donkey → ?\n6. pigeon → ?\n7. orange → ?\n8. banana → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"early\", \"2\": \"late\", \"3\": \"early\", \"4\": \"early\", \"5\": \"animal\", \"6\": \"animal\", \"7\": \"food\", \"8\": \"food\"}}"
 },
 {
  "task_id": "shifting_medium_009",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. walnut → ?\n2. penguin → ?\n3. pepper → ?\n4. needle → ?\n\nRule change: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n5. pigeon → ?\n6. candle → ?\n7. pasta → ?\n8. orange → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"food\", \"2\": \"animal\", \"3\": \"food\", \"4\": \"object\", \"5\": \"late\", \"6\": \"early\", \"7\": \"late\", \"8\": \"late\"}}"
 },
 {
  "task_id": "shifting_medium_010",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. walnut → ?\n2. mirror → ?\n3. needle → ?\n4. shovel → ?\n\nRule change: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n5. basket → ?\n6. penguin → ?\n7. turnip → ?\n8. mango → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"food\", \"2\": \"object\", \"3\": \"object\", \"4\": \"object\", \"5\": \"early\", \"6\": \"late\", \"7\": \"late\", \"8\": \"early\"}}"
 },
 {
  "task_id": "shifting_medium_011",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. eagle → ?\n2. wrench → ?\n3. turnip → ?\n4. basket → ?\n\nRule change: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n5. candle → ?\n6. mango → ?\n7. pigeon → ?\n8. cherry → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"animal\", \"2\": \"object\", \"3\": \"food\", \"4\": \"object\", \"5\": \"early\", \"6\": \"early\", \"7\": \"late\", \"8\": \"early\"}}"
 },
 {
  "task_id": "shifting_medium_012",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. pigeon → ?\n2. mirror → ?\n3. giraffe → ?\n4. pepper → ?\n\nRule change: Classify each word by CATEGORY: animal, food, or object\n\n5. pencil → ?\n6. needle → ?\n7. cherry → ?\n8. basket → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"late\", \"2\": \"early\", \"3\": \"early\", \"4\": \"late\", \"5\": \"object\", \"6\": \"object\", \"7\": \"food\", \"8\": \"object\"}}"
 },
 {
  "task_id": "shifting_medium_013",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. candle → ?\n2. parrot → ?\n3. falcon → ?\n4. mango → ?\n\nRule change: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n5. shovel → ?\n6. walnut → ?\n7. donkey → ?\n8. wrench → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"object\", \"2\": \"animal\", \"3\": \"animal\", \"4\": \"food\", \"5\": \"late\", \"6\": \"late\", \"7\": \"early\", \"8\": \"late\"}}"
 },
 {
  "task_id": "shifting_medium_014",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. tiger → ?\n2. basket → ?\n3. parrot → ?\n4. giraffe → ?\n\nRule change: Classify each word by CATEGORY: animal, food, or object\n\n5. walnut → ?\n6. pigeon → ?\n7. pasta → ?\n8. pepper → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"late\", \"2\": \"early\", \"3\": \"late\", \"4\": \"early\", \"5\": \"food\", \"6\": \"animal\", \"7\": \"food\", \"8\": \"food\"}}"
 },
 {
  "task_id": "shifting_medium_015",
  "task_type": "shifting",
  "difficulty": "Medium",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. mango → ?\n2. cherry → ?\n3. tiger → ?\n4. walnut → ?\n\nRule change: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n5. falcon → ?\n6. turnip → ?\n7. pepper → ?\n8. pencil → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"food\", \"2\": \"food\", \"3\": \"animal\", \"4\": \"food\", \"5\": \"early\", \"6\": \"late\", \"7\": \"late\", \"8\": \"late\"}}"
 },
 {
  "task_id": "shifting_hard_016",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. penguin → ?\n2. basket → ?\n3. orange → ?\n4. wrench → ?\n5. shovel → ?\n\n(Note: from this point, classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more)\n\n6. pigeon → ?\n7. mango → ?\n8. needle → ?\n9. giraffe → ?\n10. pasta → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"animal\", \"2\": \"object\", \"3\": \"food\", \"4\": \"object\", \"5\": \"object\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_hard_017",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. pepper → ?\n2. eagle → ?\n3. wrench → ?\n4. banana → ?\n5. penguin → ?\n\n(Note: from this point, classify each word by category: animal, food, or object)\n\n6. cherry → ?\n7. shovel → ?\n8. hammer → ?\n9. pigeon → ?\n10. mirror → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"2\", \"2\": \"2\", \"3\": \"1\", \"4\": \"3+\", \"5\": \"2\", \"6\": \"food\", \"7\": \"object\", \"8\": \"object\", \"9\": \"animal\", \"10\": \"object\"}}"
 },
 {
  "task_id": "shifting_hard_018",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. parrot → ?\n2. turnip → ?\n3. pigeon → ?\n4. banana → ?\n5. penguin → ?\n\n(Note: from this point, classify each word by category: animal, food, or object)\n\n6. candle → ?\n7. pepper → ?\n8. giraffe → ?\n9. orange → ?\n10. mirror → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"2\", \"2\": \"2\", \"3\": \"2\", \"4\": \"3+\", \"5\": \"2\", \"6\": \"object\", \"7\": \"food\", \"8\": \"animal\", \"9\": \"food\", \"10\": \"object\"}}"
 },
 {
  "task_id": "shifting_hard_019",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. mango → ?\n2. penguin → ?\n3. turnip → ?\n4. tiger → ?\n5. needle → ?\n\n(Note: from this point, classify each word by category: animal, food, or object)\n\n6. basket → ?\n7. pencil → ?\n8. eagle → ?\n9. banana → ?\n10. pepper → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"2\", \"2\": \"2\", \"3\": \"2\", \"4\": \"2\", \"5\": \"2\", \"6\": \"object\", \"7\": \"object\", \"8\": \"animal\", \"9\": \"food\", \"10\": \"food\"}}"
 },
 {
  "task_id": "shifting_hard_020",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. giraffe → ?\n2. shovel → ?\n3. tiger → ?\n4. hammer → ?\n5. basket → ?\n\n(Note: from this point, classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more)\n\n6. walnut → ?\n7. mirror → ?\n8. pigeon → ?\n9. pasta → ?\n10. candle → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"animal\", \"2\": \"object\", \"3\": \"animal\", \"4\": \"object\", \"5\": \"object\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_hard_021",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. candle → ?\n2. falcon → ?\n3. shovel → ?\n4. mango → ?\n5. pepper → ?\n\n(Note: from this point, classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more)\n\n6. eagle → ?\n7. basket → ?\n8. mirror → ?\n9. parrot → ?\n10. tiger → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"object\", \"2\": \"animal\", \"3\": \"object\", \"4\": \"food\", \"5\": \"food\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_hard_022",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. donkey → ?\n2. pepper → ?\n3. giraffe → ?\n4. hammer → ?\n5. orange → ?\n\n(Note: from this point, classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more)\n\n6. falcon → ?\n7. parrot → ?\n8. cherry → ?\n9. mirror → ?\n10. banana → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"animal\", \"2\": \"food\", \"3\": \"animal\", \"4\": \"object\", \"5\": \"food\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"3+\"}}"
 },
 {
  "task_id": "shifting_hard_023",
  "task_type": "shifting",
  "difficulty": "Hard",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. eagle → ?\n2. giraffe → ?\n3. walnut → ?\n4. pigeon → ?\n5. donkey → ?\n\n(Note: from this point, classify each word by category: animal, food, or object)\n\n6. orange → ?\n7. parrot → ?\n8. tiger → ?\n9. banana → ?\n10. pencil → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"2\", \"2\": \"2\", \"3\": \"2\", \"4\": \"2\", \"5\": \"2\", \"6\": \"food\", \"7\": \"animal\", \"8\": \"animal\", \"9\": \"food\", \"10\": \"object\"}}"
 },
 {
  "task_id": "shifting_expert_024",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. wrench → ?\n2. pencil → ?\n3. parrot → ?\n4. shovel → ?\n5. needle → ?\n\nContinue with the following adjustment — classify each word by first letter: 'early' if a-m, 'late' if n-z\n\n6. pasta → ?\n7. mirror → ?\n8. tiger → ?\n9. walnut → ?\n10. banana → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"1\", \"2\": \"2\", \"3\": \"2\", \"4\": \"2\", \"5\": \"2\", \"6\": \"late\", \"7\": \"early\", \"8\": \"late\", \"9\": \"late\", \"10\": \"early\"}}"
 },
 {
  "task_id": "shifting_expert_025",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. mirror → ?\n2. needle → ?\n3. walnut → ?\n4. tiger → ?\n5. donkey → ?\n\nContinue with the following adjustment — classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. eagle → ?\n7. turnip → ?\n8. wrench → ?\n9. basket → ?\n10. mango → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"early\", \"2\": \"late\", \"3\": \"late\", \"4\": \"late\", \"5\": \"early\", \"6\": \"2\", \"7\": \"2\", \"8\": \"1\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_expert_026",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. pepper → ?\n2. pencil → ?\n3. pasta → ?\n4. orange → ?\n5. mirror → ?\n\nContinue with the following adjustment — classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. basket → ?\n7. hammer → ?\n8. tiger → ?\n9. shovel → ?\n10. eagle → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"late\", \"2\": \"late\", \"3\": \"late\", \"4\": \"late\", \"5\": \"early\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_expert_027",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. walnut → ?\n2. shovel → ?\n3. wrench → ?\n4. turnip → ?\n5. orange → ?\n\nContinue with the following adjustment — classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. needle → ?\n7. candle → ?\n8. pencil → ?\n9. donkey → ?\n10. giraffe → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"late\", \"2\": \"late\", \"3\": \"late\", \"4\": \"late\", \"5\": \"late\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_expert_028",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. donkey → ?\n2. pepper → ?\n3. parrot → ?\n4. orange → ?\n5. cherry → ?\n\nContinue with the following adjustment — classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. eagle → ?\n7. wrench → ?\n8. tiger → ?\n9. pasta → ?\n10. pigeon → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"early\", \"2\": \"late\", \"3\": \"late\", \"4\": \"late\", \"5\": \"early\", \"6\": \"2\", \"7\": \"1\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_expert_029",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. pigeon → ?\n2. giraffe → ?\n3. tiger → ?\n4. needle → ?\n5. donkey → ?\n\nContinue with the following adjustment — classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. shovel → ?\n7. walnut → ?\n8. candle → ?\n9. turnip → ?\n10. pepper → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"late\", \"2\": \"early\", \"3\": \"late\", \"4\": \"late\", \"5\": \"early\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_expert_030",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. falcon → ?\n2. turnip → ?\n3. pigeon → ?\n4. wrench → ?\n5. mirror → ?\n\nContinue with the following adjustment — classify each word by syllable count: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. parrot → ?\n7. donkey → ?\n8. orange → ?\n9. banana → ?\n10. penguin → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"early\", \"2\": \"late\", \"3\": \"late\", \"4\": \"late\", \"5\": \"early\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"3+\", \"10\": \"2\"}}"
 },
 {
  "task_id": "shifting_expert_031",
  "task_type": "shifting",
  "difficulty": "Expert",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. shovel → ?\n2. giraffe → ?\n3. pepper → ?\n4. wrench → ?\n5. needle → ?\n\nContinue with the following adjustment — classify each word by first letter: 'early' if a-m, 'late' if n-z\n\n6. orange → ?\n7. pencil → ?\n8. donkey → ?\n9. banana → ?\n10. falcon → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"2\", \"2\": \"2\", \"3\": \"2\", \"4\": \"1\", \"5\": \"2\", \"6\": \"late\", \"7\": \"late\", \"8\": \"early\", \"9\": \"early\", \"10\": \"early\"}}"
 },
 {
  "task_id": "shifting_frontier_032",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. orange → ?\n2. pencil → ?\n3. walnut → ?\n4. penguin → ?\n5. tiger → ?\n\nClassify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n6. candle → ?\n7. giraffe → ?\n8. turnip → ?\n9. falcon → ?\n10. pasta → ?\n11. mango → ?\n12. needle → ?\n13. parrot → ?\n14. cherry → ?\n15. hammer → ?\n\nClassify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n16. eagle → ?\n17. pepper → ?\n18. mirror → ?\n19. pigeon → ?\n20. basket → ?\n21. donkey → ?\n22. banana → ?\n23. wrench → ?\n24. shovel → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"food\", \"2\": \"object\", \"3\": \"food\", \"4\": \"animal\", \"5\": \"animal\", \"6\": \"early\", \"7\": \"early\", \"8\": \"late\", \"9\": \"early\", \"10\": \"late\", \"11\": \"early\", \"12\": \"late\", \"13\": \"late\", \"14\": \"early\", \"15\": \"early\", \"16\": \"2\", \"17\": \"2\", \"18\": \"2\", \"19\": \"2\", \"20\": \"2\", \"21\": \"2\", \"22\": \"3+\", \"23\": \"1\", \"24\": \"2\"}}"
 },
 {
  "task_id": "shifting_frontier_033",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. basket → ?\n2. tiger → ?\n3. pencil → ?\n4. falcon → ?\n5. hammer → ?\n\nClassify each word by CATEGORY: animal, food, or object\n\n6. mirror → ?\n7. shovel → ?\n8. mango → ?\n9. giraffe → ?\n10. pepper → ?\n11. pasta → ?\n12. eagle → ?\n13. candle → ?\n14. walnut → ?\n15. penguin → ?\n\nClassify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n16. cherry → ?\n17. wrench → ?\n18. banana → ?\n19. turnip → ?\n20. needle → ?\n21. donkey → ?\n22. parrot → ?\n23. pigeon → ?\n24. orange → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"2\", \"2\": \"2\", \"3\": \"2\", \"4\": \"2\", \"5\": \"2\", \"6\": \"object\", \"7\": \"object\", \"8\": \"food\", \"9\": \"animal\", \"10\": \"food\", \"11\": \"food\", \"12\": \"animal\", \"13\": \"object\", \"14\": \"food\", \"15\": \"animal\", \"16\": \"early\", \"17\": \"late\", \"18\": \"early\", \"19\": \"late\", \"20\": \"late\", \"21\": \"early\", \"22\": \"late\", \"23\": \"late\", \"24\": \"late\"}}"
 },
 {
  "task_id": "shifting_frontier_034",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n1. falcon → ?\n2. banana → ?\n3. basket → ?\n4. tiger → ?\n5. wrench → ?\n\nClassify each word by CATEGORY: animal, food, or object\n\n6. candle → ?\n7. pigeon → ?\n8. turnip → ?\n9. donkey → ?\n10. shovel → ?\n11. hammer → ?\n12. parrot → ?\n13. cherry → ?\n14. needle → ?\n15. giraffe → ?\n\nClassify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n16. pencil → ?\n17. pasta → ?\n18. pepper → ?\n19. mirror → ?\n20. walnut → ?\n21. mango → ?\n22. eagle → ?\n23. orange → ?\n24. penguin → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"early\", \"2\": \"early\", \"3\": \"early\", \"4\": \"late\", \"5\": \"late\", \"6\": \"object\", \"7\": \"animal\", \"8\": \"food\", \"9\": \"animal\", \"10\": \"object\", \"11\": \"object\", \"12\": \"animal\", \"13\": \"food\", \"14\": \"object\", \"15\": \"animal\", \"16\": \"2\", \"17\": \"2\", \"18\": \"2\", \"19\": \"2\", \"20\": \"2\", \"21\": \"2\", \"22\": \"2\", \"23\": \"2\", \"24\": \"2\"}}"
 },
 {
  "task_id": "shifting_frontier_035",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. falcon → ?\n2. penguin → ?\n3. pencil → ?\n4. pepper → ?\n5. candle → ?\n\nClassify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. donkey → ?\n7. pasta → ?\n8. walnut → ?\n9. pigeon → ?\n10. parrot → ?\n11. cherry → ?\n12. basket → ?\n13. hammer → ?\n14. mango → ?\n15. shovel → ?\n\nClassify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n16. wrench → ?\n17. turnip → ?\n18. banana → ?\n19. orange → ?\n20. tiger → ?\n21. mirror → ?\n22. needle → ?\n23. eagle → ?\n24. giraffe → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"animal\", \"2\": \"animal\", \"3\": \"object\", \"4\": \"food\", \"5\": \"object\", \"6\": \"2\", \"7\": \"2\", \"8\": \"2\", \"9\": \"2\", \"10\": \"2\", \"11\": \"2\", \"12\": \"2\", \"13\": \"2\", \"14\": \"2\", \"15\": \"2\", \"16\": \"late\", \"17\": \"late\", \"18\": \"early\", \"19\": \"late\", \"20\": \"late\", \"21\": \"early\", \"22\": \"late\", \"23\": \"early\", \"24\": \"early\"}}"
 },
 {
  "task_id": "shifting_frontier_036",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. pasta → ?\n2. falcon → ?\n3. tiger → ?\n4. pigeon → ?\n5. eagle → ?\n\nClassify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n6. wrench → ?\n7. orange → ?\n8. donkey → ?\n9. parrot → ?\n10. cherry → ?\n11. pencil → ?\n12. walnut → ?\n13. pepper → ?\n14. basket → ?\n15. hammer → ?\n\nClassify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n16. candle → ?\n17. mango → ?\n18. needle → ?\n19. penguin → ?\n20. banana → ?\n21. shovel → ?\n22. mirror → ?\n23. turnip → ?\n24. giraffe → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"food\", \"2\": \"animal\", \"3\": \"animal\", \"4\": \"animal\", \"5\": \"animal\", \"6\": \"late\", \"7\": \"late\", \"8\": \"early\", \"9\": \"late\", \"10\": \"early\", \"11\": \"late\", \"12\": \"late\", \"13\": \"late\", \"14\": \"early\", \"15\": \"early\", \"16\": \"2\", \"17\": \"2\", \"18\": \"2\", \"19\": \"2\", \"20\": \"3+\", \"21\": \"2\", \"22\": \"2\", \"23\": \"2\", \"24\": \"2\"}}"
 },
 {
  "task_id": "shifting_frontier_037",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. mango → ?\n2. needle → ?\n3. shovel → ?\n4. pasta → ?\n5. candle → ?\n\nClassify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n6. giraffe → ?\n7. basket → ?\n8. wrench → ?\n9. pencil → ?\n10. pepper → ?\n11. pigeon → ?\n12. hammer → ?\n13. cherry → ?\n14. parrot → ?\n15. penguin → ?\n\nClassify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n16. falcon → ?\n17. orange → ?\n18. tiger → ?\n19. banana → ?\n20. donkey → ?\n21. eagle → ?\n22. walnut → ?\n23. turnip → ?\n24. mirror → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"food\", \"2\": \"object\", \"3\": \"object\", \"4\": \"food\", \"5\": \"object\", \"6\": \"2\", \"7\": \"2\", \"8\": \"1\", \"9\": \"2\", \"10\": \"2\", \"11\": \"2\", \"12\": \"2\", \"13\": \"2\", \"14\": \"2\", \"15\": \"2\", \"16\": \"early\", \"17\": \"late\", \"18\": \"late\", \"19\": \"early\", \"20\": \"early\", \"21\": \"early\", \"22\": \"late\", \"23\": \"late\", \"24\": \"early\"}}"
 },
 {
  "task_id": "shifting_frontier_038",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by CATEGORY: animal, food, or object\n\n1. pepper → ?\n2. mango → ?\n3. banana → ?\n4. penguin → ?\n5. pigeon → ?\n\nClassify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n6. shovel → ?\n7. orange → ?\n8. wrench → ?\n9. candle → ?\n10. pencil → ?\n11. basket → ?\n12. walnut → ?\n13. hammer → ?\n14. needle → ?\n15. pasta → ?\n\nClassify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n16. cherry → ?\n17. falcon → ?\n18. giraffe → ?\n19. parrot → ?\n20. mirror → ?\n21. turnip → ?\n22. donkey → ?\n23. eagle → ?\n24. tiger → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"food\", \"2\": \"food\", \"3\": \"food\", \"4\": \"animal\", \"5\": \"animal\", \"6\": \"late\", \"7\": \"late\", \"8\": \"late\", \"9\": \"early\", \"10\": \"late\", \"11\": \"early\", \"12\": \"late\", \"13\": \"early\", \"14\": \"late\", \"15\": \"late\", \"16\": \"2\", \"17\": \"2\", \"18\": \"2\", \"19\": \"2\", \"20\": \"2\", \"21\": \"2\", \"22\": \"2\", \"23\": \"2\", \"24\": \"2\"}}"
 },
 {
  "task_id": "shifting_frontier_039",
  "task_type": "shifting",
  "difficulty": "Frontier",
  "prompt": "Classify each word below according to the CURRENT rule.\nThe rule may change partway through — pay close attention.\n\nRULE: Classify each word by SYLLABLE COUNT: '1' for one syllable, '2' for two, '3+' for three or more\n\n1. donkey → ?\n2. hammer → ?\n3. walnut → ?\n4. pepper → ?\n5. banana → ?\n\nClassify each word by CATEGORY: animal, food, or object\n\n6. tiger → ?\n7. shovel → ?\n8. mirror → ?\n9. giraffe → ?\n10. candle → ?\n11. wrench → ?\n12. basket → ?\n13. pigeon → ?\n14. turnip → ?\n15. eagle → ?\n\nClassify each word by FIRST LETTER: 'early' if A-M, 'late' if N-Z\n\n16. mango → ?\n17. penguin → ?\n18. orange → ?\n19. needle → ?\n20. cherry → ?\n21. pasta → ?\n22. parrot → ?\n23. pencil → ?\n24. falcon → ?\n\nFormat your answer as:\nANSWER:\n1. [classification]\n2. [classification]\n3. [classification]\n4. [classification]\n5. [classification]\n6. [classification]\n7. [classification]\n8. [classification]\n9. [classification]\n10. [classification]\n11. [classification]\n12. [classification]\n13. [classification]\n14. [classification]\n15. [classification]\n16. [classification]\n17. [classification]\n18. [classification]\n19. [classification]\n20. [classification]\n21. [classification]\n22. [classification]\n23. [classification]\n24. [classification]\n25. [classification]",
  "gold_json": "{\"answers\": {\"1\": \"2\", \"2\": \"2\", \"3\": \"2\", \"4\": \"2\", \"5\": \"3+\", \"6\": \"animal\", \"7\": \"object\", \"8\": \"object\", \"9\": \"animal\", \"10\": \"object\", \"11\": \"object\", \"12\": \"object\", \"13\": \"animal\", \"14\": \"food\", \"15\": \"animal\", \"16\": \"early\", \"17\": \"late\", \"18\": \"late\", \"19\": \"late\", \"20\": \"early\", \"21\": \"late\", \"22\": \"late\", \"23\": \"late\", \"24\": \"early\"}}"
 }
]
''')

print(f"Loaded {len(DATASET)} items")
for tt in ['shifting']:
    count = sum(1 for d in DATASET if d["task_type"] == tt)
    print(f"  {tt}: {count} items")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4: Execution Loop
# ══════════════════════════════════════════════════════════════════════

TASK_DISPATCH = {
    "shifting": cogattention_shifting,
}

n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )

print(f"\nCompleted {n_total} items for Attention Shifting")
